In [1]:
!uv pip install --upgrade openai openai-agents python-dotenv gradio

Using Python 3.14.7 environment at: c:\Users\yoyou\OneDrive\Documents\Cursor Projects\.venv
Resolved 77 packages in 362ms
Checked 77 packages in 5ms


In [2]:
from dotenv import load_dotenv
import os
import json
from pypdf import PdfReader
from IPython.display import Markdown, display
from openai import AsyncAzureOpenAI
from agents import (
    Agent,
    Runner,
    function_tool,
    OpenAIChatCompletionsModel,
    set_tracing_disabled,
    SQLiteSession,
)
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
import gradio as gr

c:\Users\yoyou\OneDrive\Documents\Cursor Projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Azure endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("Azure API key loaded:", bool(AZURE_OPENAI_API_KEY))

Azure endpoint loaded: True
Azure API key loaded: True


In [4]:
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)

print("Azure model configured for the OpenAI Agents SDK.")

Azure model configured for the OpenAI Agents SDK.


In [5]:
from agents import Agent
from agents import function_tool
import json

MODERATION_GLOSSARY = {
    "kys": "Kill yourself (Self-harm / Harassment incitement)",
    "pos": "Piece of shit (Severe derogatory slur/insult)",
    "gnoc": "Get naked on camera (Predatory / sexually explicit solicitation)",
    "187": "Homicide / death threat slang (Penal code reference)",
    "dox": "Publish private personal information with malicious intent",
    "touch grass": "Condescending phrase telling someone to go outside (mild hostility)",
    "clown": "Dismissive insult indicating foolishness or incompetence",
    "bot": "Dehumanizing insult implying repetitive or non-sentient behavior",
    "dw": "Don't worry (benign)",
    "gg": "Good game (benign sportsmanship)",
    "afk": "Away from keyboard (neutral)",
    "rn": "Right now (neutral)",
    "tbh": "To be honest (neutral)",
    "idk": "I don't know (neutral)",
}

MODERATION_POLICIES = {
    "POL-101": {
        "title": "Direct Threats & Physical Violence",
        "description": "Explicit or veiled statements indicating intent to inflict bodily injury or self-harm incitement.",
        "severity": "CRITICAL",
        "standard_action": "FLAG",
    },
    "POL-102": {
        "title": "Targeted Harassment, Abuse & Slurs",
        "description": "Repeated unwelcome attacks, severe derogatory insults, slurs, or hostility directed at individuals.",
        "severity": "HIGH",
        "standard_action": "FLAG",
    },
    "POL-103": {
        "title": "Predatory Behavior & Inappropriate Exploitation",
        "description": "Unsolicited sexual solicitation, coercive language, or requests for explicit imagery.",
        "severity": "CRITICAL",
        "standard_action": "FLAG",
    },
    "POL-104": {
        "title": "Hostility, Mild Insults & Banter",
        "description": "Low-level bickering, sarcasm, gaming trash-talk, or slang without systematic targeting.",
        "severity": "LOW",
        "standard_action": "ALLOW",
    },
    "POL-105": {
        "title": "Ambiguous Context & Sarcasm",
        "description": "Messages containing words that could be either derogatory or benign depending entirely on context.",
        "severity": "MEDIUM",
        "standard_action": "HUMAN REVIEW",
    },
}

@function_tool
def lookup_glossary_term(term: str) -> str:
    """Look up an abbreviation, internet slang, or suspicious term in the moderation glossary.

    Args:
        term: The exact acronym or word (e.g. 'pos', 'kys', 'dw').
    """
    key = term.lower().strip()
    if key in MODERATION_GLOSSARY:
        return f"Term: '{key}' -> Definition: {MODERATION_GLOSSARY[key]}"
    return f"Term '{term}' is not listed in the moderation glossary."

content_specialist = Agent(
    name="Content Analysis Specialist",
    instructions="""You examine message text for harmful language, slurs, harassment, or hidden slang.
1. Check for acronyms or suspicious terms and look them up with lookup_glossary_term.
2. Quote exact toxic or suspicious message fragments as direct evidence.
3. Classify language into: Toxic, Ambiguous, or Benign, and state your confidence level.
Output a concise evidence summary.""",
    tools=[lookup_glossary_term],
    model=azure_model,
)

import gradio as gr
from agents import Runner

async def chat(message: str, history):
    result = await Runner.run(content_specialist, message)
    return result.final_output

gr.ChatInterface(chat).launch(inbrowser=True)





* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
